# Stopping time: Q-learning for the asset selling problem

`A_10_AssetSellingValueIteration.ipynb` solved the asset selling problem by
value iteration, using the offer distribution $p$ explicitly. Here we solve
it **without a model**: the distribution is never used by the algorithm,
only sampled from, as if offers were arriving one by one in the real world.

Because the only decision is when to stop, the whole value function
collapses to a single number: the value $\beta$ of declining the current
offer. The Bellman equation is the scalar fixed point

$$\beta = \bar\alpha\,\mathbb{E}\big[\max(w,\beta)\big],$$

with $\bar\alpha$ the discount factor and $w$ a fresh offer. The notebook
does this twice:

1. **Stochastic approximation.** Drive the recursion
   $\beta_{t+1}=(1-\gamma^t)\beta_t+\gamma^t\bar\alpha\max(w_{t+1},\beta_t)$
   with sampled offers. This is the plain averaging scheme of Unit 10,
   applied to a quantity whose own definition involves $\beta$.
2. **Q-learning.** Run simulated trajectories under an arbitrary
   exploration policy -- here a random buyer who accepts with a probability
   that grows with the offer -- and update the same scalar from the
   observed transitions.

Both should land on the value iteration answer, $\beta^\star=16\,857.9$,
whose threshold is $16\,900$ euros.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (14, 4)

from q_learning import sample_dist

rng = np.random.default_rng(0)

In [ ]:
C        = 101
alpha    = 0.5        # step size
alphabar = 0.99       # discount factor
c        = 10000 + np.arange(C) * 100
var      = 2000.0 ** 2
p        = np.exp(-1 / (2 * var) * (c - c[C // 2]) ** 2)
p        = p / p.sum()

# the answer we are aiming at, computed WITH the model for reference only
beta_star = 15000.0
for _ in range(1000):
    beta_star = alphabar * np.sum(p * np.maximum(c, beta_star))
print(f"fixed point of the Bellman equation: {beta_star:.1f}")
print(f"corresponding threshold:             {c[np.argmax(c >= beta_star)]} euros")

In [ ]:
# 1. stochastic approximation
Niter = 100_000
beta  = np.zeros(Niter + 1)
x     = np.zeros(Niter + 1)
beta[0] = c[C // 2]

for ell in range(1, Niter + 1):
    x[ell]  = c[sample_dist(p, rng)]
    gamma   = alpha if ell <= 1000 else alpha / (ell / 1000)   # decaying step size
    beta[ell] = (1 - gamma) * beta[ell - 1] + gamma * alphabar * max(x[ell], beta[ell - 1])

print(f"beta -> {beta[-1]:.1f}   (exact {beta_star:.1f})")

fig, ax = plt.subplots(1, 2)
ax[0].plot(beta);    ax[0].axhline(beta_star, color="r", ls="--", lw=1)
ax[0].set_xlabel("iteration number"); ax[0].set_ylabel(r"$\beta$")
ax[1].plot(x[1:], ",")
ax[1].set_xlabel("iteration number"); ax[1].set_ylabel("sampled offer")
plt.tight_layout(); plt.show()

In [ ]:
# 2. Q-learning, with an arbitrary exploration policy
def model_and_cost(x, u):
    """Simulator. u = 1 accepts the offer x and stops; u = 0 draws a new one."""
    if u == 1:
        return -1.0, x          # terminal; the "cost" collected is the price
    return c[sample_dist(p, rng)], 0.0


nsim  = 300_000
Q     = 0.0
Qhist = np.zeros(nsim)

for isim in range(nsim):
    gamma = 0.95 if isim < 1000 else 0.95 / (isim / 1000)
    x_k = c[sample_dist(np.ones(C) / C, rng)]
    while True:
        # exploration policy: accept with probability proportional to the offer.
        # Any policy would do -- Q-learning is off-policy.
        u_k = 1 if rng.random() < x_k / (2 * 20000) else 0
        x_next, _ = model_and_cost(x_k, u_k)
        if u_k == 1:
            break
        Q   = (1 - gamma) * Q + gamma * alphabar * max(Q, x_next)
        x_k = x_next
    Qhist[isim] = Q

print(f"Q -> {Q:.1f}   (exact {beta_star:.1f})")
print(f"learned threshold: {c[np.argmax(c >= Q)]} euros"
      f"   (exact {c[np.argmax(c >= beta_star)]})")

plt.figure(figsize=(14, 3))
plt.plot(Qhist); plt.axhline(beta_star, color="r", ls="--", lw=1)
plt.xlabel("simulation number"); plt.ylabel("Q"); plt.show()

Both routes land within a grid step of the model-based answer, and the
learned threshold is the same $16\,900$ euros (give or take one $100$-euro
step) that value iteration produced -- obtained without ever writing down
the offer distribution.

The cost is visible in the plots: hundreds of thousands of samples to pin
down a *single number*. That is the data inefficiency the unit warns about,
on a problem whose model-based solution takes sixty iterations of a
one-line recursion.